In [18]:
import os, json, time
from dotenv import load_dotenv

load_dotenv()

PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

print("PageIndex API KEY Loaded:", PAGEINDEX_API_KEY[:6])
print("OpenAI API KEY Loaded:", OPENAI_API_KEY[:6])

PageIndex API KEY Loaded: ea054f
OpenAI API KEY Loaded: sk-pro


In [16]:
from pageindex import PageIndexClient
from openai import OpenAI

PI_CLIENT = PageIndexClient(api_key=PAGEINDEX_API_KEY)
OPENAI_CLIENT = OpenAI(api_key=OPENAI_API_KEY)

print("PageIndex Client Initialized")
print("OpenAI Client Initialized")

PageIndex Client Initialized
OpenAI Client Initialized


In [30]:
from pathlib import Path
BASE_DIR = Path.cwd().parent

PDF_PATH = (
    BASE_DIR
    / "Data"
    / "StudyMaterial"
    / "Complete AI"
    / "AI Agents guidebook.pdf"
)

print(PDF_PATH)

c:\Project\AI\Viru_AI_Hub\Data\StudyMaterial\Complete AI\AI Agents guidebook.pdf


In [ ]:
result = PI_CLIENT.submit_document(PDF_PATH)

print("Document Submitted")
print(result)
doc_id = result["doc_id"]


Document Submitted
{'doc_id': 'pi-cmps9bjmz00hr01quulhissiy'}


In [33]:
while True:
    status_result = PI_CLIENT.get_document(doc_id)
    print(status_result)
    status = status_result.get("status")
    print(status)
    if status == "completed":
        print("Document indexed successfully")
        break
    elif status == "failed":
        print("Document submission failed")
        break
    time.sleep(5)

{'id': 'pi-cmps9bjmz00hr01quulhissiy', 'name': 'AI Agents guidebook.pdf', 'description': 'This illustrated guidebook provides a comprehensive framework for understanding AI agents, covering their core building blocks, design patterns, and maturity levels, while offering 12 hands-on projects for building practical agentic applications using tools like CrewAI, Ollama, and MCP.', 'status': 'completed', 'createdAt': '2026-05-30T11:17:56.685000', 'pageNum': 117, 'folderId': None}
completed
Document indexed successfully


In [52]:
tree_result = PI_CLIENT.get_tree(doc_id, node_summary=True)
print(tree_result)
pageIndex_tree = tree_result["result"]

{'doc_id': 'pi-cmps9bjmz00hr01quulhissiy', 'status': 'completed', 'retrieval_ready': True, 'result': [{'title': 'AI AGENTS', 'node_id': '0000', 'page_index': 1, 'prefix_summary': '# AI AGENTS\n', 'text': '# AI AGENTS\n', 'nodes': [{'title': 'THE ILLUSTRATED GUIDEBOOK', 'node_id': '0001', 'page_index': 1, 'summary': 'This document serves as an illustrated guidebook on AI Agents by Avi Chawla and Akshay Pachaar. It provides a diagnostic assessment to tailor the reading experience, defines fundamental concepts like LLMs and RAG, outlines the essential building blocks and design patterns of agentic systems, categorizes agent maturity levels, and details 12 hands-on projects for building practical AI Agent applications.', 'text': "## THE ILLUSTRATED GUIDEBOOK\n\n![img-0.jpeg](img-0.jpeg)\n![img-1.jpeg](img-1.jpeg)\n![img-2.jpeg](img-2.jpeg)\n![img-3.jpeg](img-3.jpeg)\n\nDaily Dose of Data Science\nAvi Chawla &amp; Akshay Pachaar DailyDoseofDS.com\n\nDailyDoseofDS.com\n\n### How to make the 

In [ ]:
def llm_tree_search(query: str, tree: list, model: str = "gpt-4o-mini") -> dict:
    """
    Core PageIndex retrieval:
    sends the query + document tree to an LLM.
    LLM reasons over the structure and returns relevant node_ids.

    Returns: dict with 'thinking' (reasoning) and 'node_list' (node IDs)
    """
    def compress(nodes):
        out = []
        for n in nodes:
            entry = {
                "node_id": n["node_id"],
                "title": n["title"],
                "page": n.get("page_index", "?"),
                "summary": n.get("text", "")[:150],
            }
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out
    
    compressed_tree = compress(tree)

    prompt = f"""You are given a query and a document's tree structure (like a Table of Contents).
    Your task: identify which node IDs most likely contain the answer to the query.
    Think step-by-step and reason through the query.

    Query: {query}
    Document Tree: 
    {json.dumps(compressed_tree, indent=2)}

    Reply ONLY in this exact JSON format:
    {{
        "thinking": "<your step-by-step reasoning>",
        "node_list": ["node_id1", "node_id2"]
    }}
    """

    response = OPENAI_CLIENT.chat.completions.create(
        model=model,
        messages=[
            {"role": "user", "content": prompt},
        ],
        response_format={"type": "json_object"},
    )

    return json.loads(response.choices[0].message.content)
                

In [ ]:
query = "What is guardrails and how to use them?"
result = llm_tree_search(query, pageIndex_tree)

print("LLM Reasoning:")
print(result.get("thinking", "N/A"))
print()
print("Selected Nodes:", result.get("node_list", []))

LLM Reasoning:
The query asks about guardrails and how to use them in the context of AI agents. Looking at the document tree structure, it seems that the node '5) Guardrails' (node_id: '0012') directly addresses the topic of guardrails. The summary mentions that agents can go off track without constraints, suggesting that this section discusses both what guardrails are and their application in a relevant context. Therefore, this is the primary node that I would refer to for an answer to the query. Additionally, node '0005', which contains building blocks of AI Agents, may provide context for how guardrails fit within the broader framework of AI design, but it is less directly related than node '0012'. Hence, I will include both node '0012' and node '0005' as likely relevant sources for the answer.

Selected Nodes: ['0012', '0005']


In [47]:
def find_nodes_by_ids(tree: list, node_ids: list) -> list:
    """
    Recursively searches the tree for nodes with matching IDs.
    Returns a list of nodes that match the IDs.
    """
    found = []
    for node in tree:
        if node["node_id"] in node_ids:
            found.append(node)
        if node.get("nodes"):
            found.extend(find_nodes_by_ids(node["nodes"], node_ids))
    return found
    

In [54]:
def generate_answer(
    query: str,
    tree: list,
    model: str = "gpt-4o-mini"
) -> str:
    """
    Generate an answer using the retrieved document nodes.

    Args:
        query: User question
        tree: List of selected nodes from PageIndex
        model: OpenAI model name

    Returns:
        Answer string
    """

    if not tree:
        return "No relevant information found in the document."

    # Build context from selected nodes
    context_parts = []

    for node in tree:
        context_parts.append(
            f"""
SECTION: {node.get('title', 'Unknown')}
PAGE: {node.get('page_index', '?')}

CONTENT:
{node.get('text', 'No content available')}
"""
        )

    context_str = "\n\n" + ("\n" + "-" * 80 + "\n").join(context_parts)

    prompt = f"""
You are an expert document analyst.

Your task is to answer the user's query using ONLY the provided document context.

Rules:
1. Use only information found in the context.
2. Do not make up information.
3. Cite every major claim using:
   (Section: <title>, Page: <page>)
4. If the answer is not present in the context, say so.
5. Write a clear and detailed answer.

User Query:
{query}

Document Context:
{context_str}
"""

    response = OPENAI_CLIENT.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "system",
                "content": "You answer questions using only the provided document context."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    return response.choices[0].message.content.strip()

In [50]:
def vectorless_rag(query: str, tree: list, verbose: bool = False) -> str:
    """
    Full end-to-end Page Index RAG Pipeline.

    STEP 1 LLM Tree Search: -> find the most relevant node_ids
    STEP 2 Node retrieval: -> fetch section content
    STEP 3 Answer Generation: -> produce the final answer
    """
    if verbose:
        print("Running Vectorless RAG Pipeline...")

    # STEP 1: LLM Tree Search
    search_result = llm_tree_search(query, tree)
    node_ids = search_result.get("node_list", [])
    if verbose:
        print("LLM Reasoning:")
        print(search_result.get("thinking", "N/A"))
        print()
        print("Selected Nodes:", search_result.get("node_list", []))

    # STEP 2: Node retrieval
    nodes = find_nodes_by_ids(tree, node_ids)

    if verbose:
        print("Retrieved Nodes:")
        for node in nodes:
            print(f"Node ID: {node['node_id']}")
            print(f"Title: {node['title']}")
            print(f"Page: {node.get('page_index', '?')}")
            print(f"Summary: {node.get('text', '')[:150]}")

    # STEP 3: Answer Generation
    answer = generate_answer(query, nodes)
    if verbose:
        print("Generated Answer:")
        print(answer)

    return answer

In [55]:
query = "What is guardrails and how to use them?"
vectorless_rag(query, pageIndex_tree)

"Guardrails are constraints implemented in AI agents to ensure they remain on track and maintain quality standards. They help prevent issues such as hallucinations, endless loops, or poor decision-making by providing boundaries within which the agents operate (Section: 5) Guardrails, Page: 23).\n\nTo use guardrails effectively, one can implement several strategies:\n\n1. **Limiting Tool Usage**: This involves preventing the agent from overusing APIs or generating irrelevant queries, ensuring that its actions remain focused and relevant.\n\n2. **Setting Validation Checkpoints**: This means establishing criteria that the agent's outputs must meet before proceeding to the next step, which helps maintain the quality of the results.\n\n3. **Establishing Fallback Mechanisms**: If an agent fails to complete a task, having another agent or a human reviewer ready to intervene can help correct the course of action.\n\nFor instance, in the case of an AI-powered legal assistant, guardrails would e